# Tratamento e exportação dos dados — Future Jobs & Skills Demand 2025

Notebook usado para gerar o dataset tratado que o `app.py` lê (`data/clean/dataset.csv`).
A análise completa está no `eda.ipynb` (EDA da Aula 11). Aqui fica só o necessário para o dashboard:
baixar o dado, salvar o bruto, criar as variáveis numéricas e exportar.

In [1]:
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd

# o notebook pode ser aberto de dentro de notebooks/ ou da raiz do projeto
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PASTA_RAW = RAIZ / "data" / "raw"
PASTA_CLEAN = RAIZ / "data" / "clean"
PASTA_RAW.mkdir(parents=True, exist_ok=True)
PASTA_CLEAN.mkdir(parents=True, exist_ok=True)

/usr/local/python/3.14.2/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Download e cópia do dado bruto

In [2]:
path = kagglehub.dataset_download("ahsanneural/future-jobs-and-skills-demand-2025")
df_original = pd.read_csv(Path(path) / "future_jobs_dataset.csv")

# guarda o bruto em data/raw (essa pasta não vai pro git)
df_original.to_csv(PASTA_RAW / "future_jobs_dataset.csv", index=False)

print(df_original.shape)
df_original.head()

  0%|                                                                                   | 0.00/139k [00:00<?, ?B/s]

100%|███████████████████████████████████████████████████████████████████████████| 139k/139k [00:00<00:00, 6.90MB/s]

Extracting files...


(10000, 9)


,job_id,job_title,industry,location,salary_usd,skills_required,remote_option,company_size,posting_date
0,1,Quantum Researcher,Quantum Computing,Singapore,175780,"Linear Algebra, Quantum Algorithms",No,Large,2025-07-22
1,2,Renewable Energy Engineer,Green Tech,Singapore,137481,"Climate Data Analysis, Energy Modeling",Yes,Large,2025-09-26
2,3,Quantum Researcher,Quantum Computing,Tokyo,182081,"Linear Algebra, Qiskit",No,Medium,2025-12-31
3,4,Sustainability Analyst,Green Tech,Singapore,113822,"Climate Data Analysis, Energy Modeling",No,Large,2025-05-29
4,5,Smart Contract Engineer,Blockchain,London,92575,"Rust, Solidity",Yes,Small,2025-03-30


In [3]:
# mesma checagem da EDA: sem nulos e sem duplicadas
print("Nulos por coluna:")
print(df_original.isna().sum())
print("\nDuplicadas:", df_original.duplicated().sum())

Nulos por coluna:
job_id             0
job_title          0
industry           0
location           0
salary_usd         0
skills_required    0
remote_option      0
company_size       0
posting_date       0
dtype: int64

Duplicadas: 0


## Por que criar novas variáveis

Na EDA vi que o dataset só tem uma variável quantitativa de verdade (`salary_usd`; o `job_id` é só identificador).
Com uma variável só não dá pra montar matriz de correlação, então criei variáveis numéricas a partir das colunas que já existiam:

| Variável nova | De onde vem | Tipo |
|---|---|---|
| `porte_empresa_num` | `company_size` (Small=1, Medium=2, Large=3) | ordinal |
| `mes_publicacao` | mês de `posting_date` (1 a 12) | discreta |
| `dia_semana_num` | dia da semana de `posting_date` (1=segunda ... 7=domingo) | discreta |
| `demanda_cargo_cidade` | quantas vagas do mesmo cargo existem na mesma cidade | contagem |

Como algumas são ordinais/discretas, a correlação usada no dashboard é a de **Spearman**, que também já era a escolha da EDA
(o salário não passou no Shapiro-Wilk).

Não usei a quantidade de skills porque toda vaga tem exatamente 2 skills, então a coluna seria constante.

In [4]:
df = df_original.copy()

df["posting_date"] = pd.to_datetime(df["posting_date"])
df["mes_publicacao"] = df["posting_date"].dt.month
df["dia_semana_num"] = df["posting_date"].dt.dayofweek + 1

df["porte_empresa_num"] = df["company_size"].map({"Small": 1, "Medium": 2, "Large": 3})

df["demanda_cargo_cidade"] = df.groupby(["job_title", "location"])["job_id"].transform("count")

df[["salary_usd", "porte_empresa_num", "mes_publicacao", "dia_semana_num", "demanda_cargo_cidade"]].describe().round(1)

,salary_usd,porte_empresa_num,mes_publicacao,dia_semana_num,demanda_cargo_cidade
count,10000.0,10000.0,10000.0,10000.0,10000.0
mean,150131.5,2.0,6.5,4.0,191.7
std,57534.9,0.8,3.5,2.0,31.9
min,50013.0,1.0,1.0,1.0,123.0
25%,100568.0,1.0,3.0,2.0,179.0
50%,150398.5,2.0,7.0,4.0,207.0
75%,199225.2,3.0,10.0,6.0,214.0
max,249990.0,3.0,12.0,7.0,228.0


## Rótulos em português para o dashboard

In [5]:
df["setor"] = df["industry"].map({
    "AI": "Inteligência Artificial",
    "Blockchain": "Blockchain",
    "Green Tech": "Tecnologia Verde",
    "Quantum Computing": "Computação Quântica",
})
df["modalidade"] = df["remote_option"].map({"Yes": "Remoto", "No": "Presencial"})
df["porte_empresa"] = df["company_size"].map({"Small": "Pequena", "Medium": "Média", "Large": "Grande"})

df["faixa_salarial"] = pd.cut(
    df["salary_usd"],
    bins=[50_000, 100_000, 150_000, 200_000, 250_000],
    labels=["50-100 mil", "100-150 mil", "150-200 mil", "200-250 mil"],
    include_lowest=True,
)

# confere se algum mapeamento ficou sem correspondência
df[["setor", "modalidade", "porte_empresa", "porte_empresa_num", "faixa_salarial"]].isna().sum()

setor                0
modalidade           0
porte_empresa        0
porte_empresa_num    0
faixa_salarial       0
dtype: int64

## Exportação

In [6]:
colunas = [
    "job_title", "setor", "location", "salary_usd", "skills_required",
    "modalidade", "porte_empresa", "posting_date",
    "porte_empresa_num", "mes_publicacao", "dia_semana_num", "demanda_cargo_cidade",
    "faixa_salarial",
]

df_clean = df[colunas].rename(columns={"job_title": "cargo", "location": "cidade", "skills_required": "skills"})
df_clean["posting_date"] = df_clean["posting_date"].dt.strftime("%Y-%m-%d")

df_clean.to_csv(PASTA_CLEAN / "dataset.csv", index=False)
print(df_clean.shape)
df_clean.head()

(10000, 13)


,cargo,setor,cidade,salary_usd,skills,modalidade,porte_empresa,posting_date,porte_empresa_num,mes_publicacao,dia_semana_num,demanda_cargo_cidade,faixa_salarial
0,Quantum Researcher,Computação Quântica,Singapore,175780,"Linear Algebra, Quantum Algorithms",Presencial,Grande,2025-07-22,3,7,2,210,150-200 mil
1,Renewable Energy Engineer,Tecnologia Verde,Singapore,137481,"Climate Data Analysis, Energy Modeling",Remoto,Grande,2025-09-26,3,9,5,202,100-150 mil
2,Quantum Researcher,Computação Quântica,Tokyo,182081,"Linear Algebra, Qiskit",Presencial,Média,2025-12-31,2,12,3,179,150-200 mil
3,Sustainability Analyst,Tecnologia Verde,Singapore,113822,"Climate Data Analysis, Energy Modeling",Presencial,Grande,2025-05-29,3,5,4,211,100-150 mil
4,Smart Contract Engineer,Blockchain,London,92575,"Rust, Solidity",Remoto,Pequena,2025-03-30,1,3,7,211,50-100 mil
